# Persiapan Data

In [ ]:
# ======================
# Setup
# ======================
from pathlib import Path
import pandas as pd
import numpy as np
import requests

# Lokasi data mentah
folder_data = Path(
    r"C:\Users\Acer\OneDrive\Pictures\Kuliah\Data Sekunder"
)

# Pastikan folder ditemukan
print("Folder ditemukan:", folder_data.exists())

In [ ]:
# ======================
# Cek File Sumber
# ======================

file_sumber = [
    "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2020.xlsx",
    "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2021.xlsx",
    "banyaknya-kecelakaan-lalu-lintas.xls",
    "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2020.csv",
    "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2021.csv",
    "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2020.csv",
    "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2021.csv"
]

for nama_file in file_sumber:
    if (folder_data / nama_file).is_file():
        print("OK:", nama_file)
    else:
        print("TIDAK DITEMUKAN:", nama_file)

In [ ]:
# Baca file 2020
df_pesawat_2020 = pd.read_excel(
    folder_data / "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2020.xlsx"
)

# Baca file 2021
df_pesawat_2021 = pd.read_excel(
    folder_data / "Jumlah Debarkasi Penumpang Penerbangan Domestik di Jawa Tengah, 2021.xlsx"
)

df_pesawat_2020

In [ ]:
# ======================
# Cleaning Data Penerbangan Domestik
# ======================

daftar_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "Nopember", "Desember"
]

def clean_pesawat(df, tahun):
    # Mengambil nama bulan dari baris ketiga
    bulan = df.iloc[2].astype(str).str.strip()

    # Mengambil kolom Januari sampai Desember
    posisi_bulan = [
        i for i, nama in enumerate(bulan)
        if nama in daftar_bulan
    ]

    assert len(posisi_bulan) == 12, f"Data bulan {tahun} tidak lengkap"

    # Membuat dataframe baru
    df_clean = pd.DataFrame({
        "bulan": bulan.iloc[posisi_bulan].values,
        "penumpang": df.iloc[3, posisi_bulan].values
    })

    # Menambahkan tahun dan mengubah tipe data
    df_clean["tahun"] = tahun
    df_clean["penumpang"] = pd.to_numeric(df_clean["penumpang"])

    return df_clean.reset_index(drop=True)


# Membersihkan data 2020 dan 2021
pesawat_2020 = clean_pesawat(df_pesawat_2020, 2020)
pesawat_2021 = clean_pesawat(df_pesawat_2021, 2021)

# Menggabungkan kedua dataframe
pesawat_clean = pd.concat(
    [pesawat_2020, pesawat_2021],
    ignore_index=True
)

display(pesawat_clean)

In [ ]:
# ======================
# Validasi Data Penerbangan
# ======================

# Memeriksa jumlah baris dan missing values
print("Jumlah baris:", len(pesawat_clean))
print("\nMissing values:")
print(pesawat_clean.isna().sum())

# Memeriksa kelengkapan bulan
print("\nJumlah bulan per tahun:")
print(pesawat_clean.groupby("tahun")["bulan"].nunique())

# Memeriksa total penumpang
print("\nTotal penumpang per tahun:")
print(pesawat_clean.groupby("tahun")["penumpang"].sum())

# Validasi jumlah baris, bulan, dan duplikasi
assert len(pesawat_clean) == 24
assert pesawat_clean.isna().sum().sum() == 0
assert pesawat_clean[["tahun", "bulan"]].duplicated().sum() == 0

for tahun, df in pesawat_clean.groupby("tahun"):
    assert set(df["bulan"]) == set(daftar_bulan)

# Membandingkan hasil cleaning dengan total tahunan di Excel
for tahun, df_asli, df_clean in [
    (2020, df_pesawat_2020, pesawat_2020),
    (2021, df_pesawat_2021, pesawat_2021)
]:
    header = df_asli.iloc[2].astype(str).str.strip()
    posisi_total = [
        i for i, nama in enumerate(header)
        if nama == "Tahunan"
    ]

    assert len(posisi_total) == 1, f"Kolom Tahunan {tahun} tidak ditemukan"

    total_asli = pd.to_numeric(df_asli.iloc[3, posisi_total[0]])
    total_clean = df_clean["penumpang"].sum()

    assert total_asli == total_clean, f"Total penumpang {tahun} tidak sesuai"

    print(f"Total {tahun} sesuai dengan data asli: {total_clean:,.0f}")

print("\nValidasi berhasil!")

In [ ]:
import requests
from getpass import getpass

api_key = getpass("Masukkan API key BPS: ")

url = f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/3300/var/1793/th/120/key/{api_key}"

r = requests.get(url, timeout=30)
r.raise_for_status()

data2020 = r.json()

In [ ]:
url2 = f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/3300/var/1793/th/121/key/{api_key}"

r2 = requests.get(url2, timeout=30)
r2.raise_for_status()

data2021 = r2.json()

In [ ]:
# ======================
# Cleaning Data Angkutan Laut
# ======================

def extract_df_fixed(data_json):
    # Mengambil tahun
    tahun = int(data_json["tahun"][0]["label"])

    # Membentuk prefix kode BPS
    prefix = (
        str(data_json["vervar"][0]["val"])
        + str(data_json["var"][0]["val"])
        + str(data_json["turvar"][0]["val"])
        + str(data_json["tahun"][0]["val"])
    )

    records = []

    for item in data_json["turtahun"]:
        bulan = item["label"]
        kode = item["val"]

        # Melewati data total tahunan
        if bulan == "Tahunan":
            continue

        key = prefix + str(kode)

        records.append({
            "tahun": tahun,
            "bulan": bulan,
            "penumpang": data_json["datacontent"][key]
        })

    return pd.DataFrame(records)


# Cleaning data 2020 dan 2021
laut_2020 = extract_df_fixed(data2020)
laut_2021 = extract_df_fixed(data2021)

# Menggabungkan data
debarkasi_angkutan_laut = pd.concat(
    [laut_2020, laut_2021],
    ignore_index=True
)

display(debarkasi_angkutan_laut)

In [ ]:
# ======================
# Validasi Data Angkutan Laut
# ======================

# Memeriksa jumlah baris dan missing values
print("Jumlah baris:", len(debarkasi_angkutan_laut))

print("\nMissing values:")
print(debarkasi_angkutan_laut.isna().sum())

print("\nJumlah bulan per tahun:")
print(debarkasi_angkutan_laut.groupby("tahun")["bulan"].nunique())

print("\nTotal penumpang per tahun:")
print(debarkasi_angkutan_laut.groupby("tahun")["penumpang"].sum())

# Validasi data
assert len(debarkasi_angkutan_laut) == 24
assert debarkasi_angkutan_laut.isna().sum().sum() == 0
assert debarkasi_angkutan_laut[["tahun", "bulan"]].duplicated().sum() == 0

for tahun, df in debarkasi_angkutan_laut.groupby("tahun"):
    assert set(df["bulan"]) == set(daftar_bulan)

# Membandingkan total bulanan dengan total tahunan BPS
for data_json, df in [
    (data2020, laut_2020),
    (data2021, laut_2021)
]:
    tahun = int(data_json["tahun"][0]["label"])

    prefix = (
        str(data_json["vervar"][0]["val"])
        + str(data_json["var"][0]["val"])
        + str(data_json["turvar"][0]["val"])
        + str(data_json["tahun"][0]["val"])
    )

    total_tahunan = data_json["datacontent"][prefix + "102"]

    assert df["penumpang"].sum() == total_tahunan, (
        f"Total penumpang {tahun} tidak sesuai"
    )

    print(f"Total {tahun} sesuai dengan BPS: {total_tahunan:,}")

print("\nValidasi berhasil!")

In [ ]:
# ======================
# Mengambil Data Kecelakaan Lalu Lintas
# ======================

file_laka = folder_data / "banyaknya-kecelakaan-lalu-lintas.xls"

# Memeriksa sheet yang tersedia
excel_laka = pd.ExcelFile(file_laka)

print("Sheet tersedia:", excel_laka.sheet_names)

# Membaca data asli tanpa mengubah header
df_raw = pd.read_excel(
    file_laka,
    sheet_name=0,
    header=None
)

print("Ukuran data:", df_raw.shape)

display(df_raw.head(12))

In [ ]:
# ======================
# Menyeleksi Data Kecelakaan
# ======================

# df_raw merupakan hasil pd.read_excel(..., header=None)

# Ambil data setelah baris header
df_data = df_raw.iloc[7:].copy()

# Ambil hanya baris yang memiliki kode wilayah
mask_wilayah = (
    df_data.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.match(r"^\d{1,2}\.$")
)

df_data = df_data.loc[mask_wilayah].reset_index(drop=True)

# Ambil kode wilayah sebagai penanda kabupaten/kota
df_data["kode_wilayah"] = (
    df_data.iloc[:, 0]
    .astype(str)
    .str.replace(".", "", regex=False)
    .astype(int)
)

# Memastikan kode wilayah lengkap
assert sorted(df_data["kode_wilayah"].tolist()) == list(range(1, 36))

print("Jumlah wilayah:", len(df_data))
display(df_data[[0, 1, "kode_wilayah"]])

In [ ]:
# ======================
# Membuat DataFrame Kecelakaan
# ======================

df_clean = pd.DataFrame({
    "kabupaten_kota": df_data.iloc[:, 1],

    "jumlah_kecelakaan_2020": df_data.iloc[:, 5],
    "jumlah_kecelakaan_2021": df_data.iloc[:, 6],

    "meninggal_2020": df_data.iloc[:, 9],
    "meninggal_2021": df_data.iloc[:, 10],

    "luka_berat_2020": df_data.iloc[:, 13],
    "luka_berat_2021": df_data.iloc[:, 14],

    "luka_ringan_2020": df_data.iloc[:, 17],
    "luka_ringan_2021": df_data.iloc[:, 18],

    "kerugian_2020": df_data.iloc[:, 21],
    "kerugian_2021": df_data.iloc[:, 22]
})

# Menyimpan kode wilayah sementara untuk cleaning
df_clean["kode_wilayah"] = df_data["kode_wilayah"]

display(df_clean.head())

In [ ]:
# ======================
# Cleaning Data Kecelakaan
# ======================

df_clean2 = df_clean.copy()

# Membersihkan awalan nama wilayah
df_clean2["kabupaten_kota"] = (
    df_clean2["kabupaten_kota"]
    .astype(str)
    .str.replace(
        r"^(ResTabes|ResTa|Res)\.?\s*",
        "",
        regex=True
    )
    .str.strip()
)

# Membedakan kabupaten dan kota berdasarkan kode wilayah
# Kode 01-29: Kabupaten
# Kode 30-35: Kota
mask_kota = df_clean2["kode_wilayah"].between(30, 35)

df_clean2.loc[mask_kota, "kabupaten_kota"] = (
    "Kota " + df_clean2.loc[mask_kota, "kabupaten_kota"]
)

# Mengubah kolom indikator menjadi numerik
kolom_nilai = [
    col for col in df_clean2.columns
    if col not in ["kabupaten_kota", "kode_wilayah"]
]

for col in kolom_nilai:
    df_clean2[col] = pd.to_numeric(
        df_clean2[col],
        errors="raise"
    )

# Konversi kerugian dari ribu rupiah ke juta rupiah
for col in ["kerugian_2020", "kerugian_2021"]:
    df_clean2[col] = df_clean2[col] / 1_000_000
# Hapus kode wilayah sementara
df_clean2 = df_clean2.drop(columns=["kode_wilayah"])

# Data akhir
laka_lantas = df_clean2.reset_index(drop=True)

display(laka_lantas)

In [ ]:
# ======================
# Validasi Data Kecelakaan
# ======================

print("Jumlah baris:", len(laka_lantas))
print("Jumlah wilayah:", laka_lantas["kabupaten_kota"].nunique())

print("\nMissing values:")
print(laka_lantas.isna().sum())

print("\nDaftar kota:")
print(laka_lantas.loc[
    laka_lantas["kabupaten_kota"].str.startswith("Kota "),
    "kabupaten_kota"
].tolist())

# Validasi jumlah wilayah dan kolom
assert laka_lantas.shape == (35, 11)

# Validasi nama wilayah
assert laka_lantas["kabupaten_kota"].is_unique
assert laka_lantas["kabupaten_kota"].notna().all()

# Validasi jumlah kabupaten dan kota
assert (
    laka_lantas["kabupaten_kota"].str.startswith("Kota ").sum()
    == 6
)

# Memastikan empat pasangan wilayah tetap berbeda
for nama in ["Magelang", "Semarang", "Pekalongan", "Tegal"]:
    assert nama in laka_lantas["kabupaten_kota"].values
    assert f"Kota {nama}" in laka_lantas["kabupaten_kota"].values

# Validasi indikator
assert laka_lantas[kolom_nilai].notna().all().all()
assert (laka_lantas[kolom_nilai] >= 0).all().all()

print("\nValidasi berhasil!")

In [ ]:
# ======================
# Mengambil Data Panjang Jalan
# ======================

file_jalan_2020 = folder_data / "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2020.csv"

file_jalan_2021 = folder_data / "Panjang Jalan Menurut Kabupaten_Kota dan Tingkat Kewenangan Pemerintahan di Provinsi Jawa Tengah (km), 2021.csv"

# Membaca data asli
jalan_2020 = pd.read_csv(
    file_jalan_2020,
    sep=None,
    engine="python",
    header=None
)

jalan_2021 = pd.read_csv(
    file_jalan_2021,
    sep=None,
    engine="python",
    header=None
)

print("Ukuran data 2020:", jalan_2020.shape)
print("Ukuran data 2021:", jalan_2021.shape)

display(jalan_2020.head(12))
display(jalan_2021.head(12))

In [ ]:
# ======================
# Cleaning Data Panjang Jalan
# ======================

def clean_jalan(df, tahun):
    # Mengambil 35 baris kabupaten/kota setelah header
    df_clean = df.iloc[1:36, :5].copy()

    # Mengubah nama kolom
    df_clean.columns = [
        "kabupaten_kota",
        "jalan_negara",
        "jalan_provinsi",
        "jalan_kabupaten_kota",
        "total_panjang_jalan"
    ]

    # Membersihkan nama wilayah
    df_clean["kabupaten_kota"] = (
        df_clean["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Mengubah kolom panjang jalan menjadi numerik
    kolom_nilai = [
        "jalan_negara",
        "jalan_provinsi",
        "jalan_kabupaten_kota",
        "total_panjang_jalan"
    ]

    for col in kolom_nilai:
        df_clean[col] = pd.to_numeric(
            df_clean[col],
            errors="coerce"
        )

    # Menambahkan tahun
    df_clean["tahun"] = tahun

    return df_clean.reset_index(drop=True)


# Cleaning data 2020 dan 2021
jalan_clean_2020 = clean_jalan(jalan_2020, 2020)
jalan_clean_2021 = clean_jalan(jalan_2021, 2021)

# Menggabungkan data
jalan_clean = pd.concat(
    [jalan_clean_2020, jalan_clean_2021],
    ignore_index=True
)

display(jalan_clean)

In [ ]:
# ======================
# Validasi Data Panjang Jalan
# ======================

print("Jumlah baris:", len(jalan_clean))

print("\nJumlah data per tahun:")
print(jalan_clean.groupby("tahun").size())

print("\nJumlah wilayah unik:")
print(jalan_clean.groupby("tahun")["kabupaten_kota"].nunique())

print("\nMissing values:")
print(jalan_clean.isna().sum())

# Memeriksa enam kota di bagian akhir
print("\nWilayah bagian akhir:")
display(jalan_clean.groupby("tahun").tail(6))

# Memeriksa kecocokan total panjang jalan
jalan_clean["total_hitung"] = jalan_clean[
    ["jalan_negara", "jalan_provinsi", "jalan_kabupaten_kota"]
].sum(axis=1, min_count=1)

jalan_clean["selisih"] = (
    jalan_clean["total_panjang_jalan"]
    - jalan_clean["total_hitung"]
)

print("\nData dengan selisih lebih dari 1 km:")
display(jalan_clean.loc[jalan_clean["selisih"].abs() > 1])

# Validasi jumlah wilayah
assert len(jalan_clean) == 70
assert (jalan_clean.groupby("tahun").size() == 35).all()
assert (jalan_clean.groupby("tahun")["kabupaten_kota"].nunique() == 35).all()

# Validasi data
assert jalan_clean["kabupaten_kota"].notna().all()
assert jalan_clean["total_panjang_jalan"].notna().all()
assert (jalan_clean["total_panjang_jalan"] > 0).all()

assert not jalan_clean.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

print("\nValidasi berhasil!")

In [ ]:
# ======================
# Memeriksa Ketidaksesuaian Data Panjang Jalan
# ======================

# Memeriksa data Kota Pekalongan dari CSV asli
print("Data asli 2020:")
display(
    jalan_2020.loc[
        jalan_2020[0].astype(str).str.contains(
            "Pekalongan", case=False, na=False
        )
    ]
)

# Membandingkan data hasil cleaning
print("\nData hasil cleaning:")
display(
    jalan_clean.loc[
        jalan_clean["kabupaten_kota"].str.contains(
            "Pekalongan", case=False, na=False
        )
    ]
)

# Memeriksa baris dengan komponen jalan yang kosong
print("\nData dengan missing values:")
display(
    jalan_clean.loc[
        jalan_clean[["jalan_negara", "jalan_provinsi"]]
        .isna().any(axis=1)
    ]
)

In [ ]:
# ======================
# Mengambil Data Kendaraan Bermotor
# ======================

file_kendaraan_2020 = folder_data / "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2020.csv"
file_kendaraan_2021 = folder_data / "Jumlah Kendaraan Bermotor Menurut Kabupaten_Kota dan Jenis Kendaraan di Provinsi Jawa Tengah (unit), 2021.csv"

# Membaca data asli
kendaraan_2020 = pd.read_csv(file_kendaraan_2020)
kendaraan_2021 = pd.read_csv(file_kendaraan_2021)

print("Ukuran data 2020:", kendaraan_2020.shape)
print("Ukuran data 2021:", kendaraan_2021.shape)

display(kendaraan_2020.head(10))
display(kendaraan_2020.tail(10))

In [ ]:
# ======================
# Cleaning Data Kendaraan Bermotor
# ======================

# Mengubah nama kolom
rename_cols = {
    "Kabupaten/Kota": "kabupaten_kota",
    "Jumlah Kendaraan Bermotor - Mobil Penumpang": "mobil_penumpang",
    "Jumlah Kendaraan Bermotor - Bus": "bus",
    "Jumlah Kendaraan Bermotor - Truk": "truk",
    "Jumlah Kendaraan Bermotor - Sepeda Motor": "sepeda_motor",
    "Jumlah Kendaraan Bermotor - Jumlah": "total"
}

# Kolom yang berisi jumlah kendaraan
kolom_nilai = [
    "mobil_penumpang",
    "bus",
    "truk",
    "sepeda_motor",
    "total"
]

# Mengambil daftar wilayah dari jalan_clean
daftar_wilayah = set(
    jalan_clean["kabupaten_kota"].astype(str).str.strip()
)

def clean_kendaraan(df, tahun):
    # Membuat salinan data
    df_clean = df.copy()

    # Mengubah nama kolom
    df_clean = df_clean.rename(columns=rename_cols)

    # Membersihkan nama wilayah
    df_clean["kabupaten_kota"] = (
        df_clean["kabupaten_kota"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Mengambil hanya 35 kabupaten/kota
    df_clean = df_clean[
        df_clean["kabupaten_kota"].isin(daftar_wilayah)
    ].copy()

    # Mengubah kolom jumlah kendaraan menjadi numerik
    for col in kolom_nilai:
        df_clean[col] = pd.to_numeric(
            df_clean[col],
            errors="coerce"
        )

    # Menambahkan tahun
    df_clean["tahun"] = tahun

    # Memilih kolom yang diperlukan
    df_clean = df_clean[
        ["kabupaten_kota"] + kolom_nilai + ["tahun"]
    ]

    return df_clean.reset_index(drop=True)


# Cleaning data 2020 dan 2021
kendaraan_clean_2020 = clean_kendaraan(kendaraan_2020, 2020)
kendaraan_clean_2021 = clean_kendaraan(kendaraan_2021, 2021)

# Menggabungkan data
kendaraan_bermotor = pd.concat(
    [kendaraan_clean_2020, kendaraan_clean_2021],
    ignore_index=True
)

display(kendaraan_bermotor)

In [ ]:
# ======================
# Validasi Data Kendaraan Bermotor
# ======================

# Memeriksa jumlah data
print("Jumlah baris:", len(kendaraan_bermotor))

print("\nJumlah wilayah per tahun:")
print(
    kendaraan_bermotor.groupby("tahun")["kabupaten_kota"].nunique()
)

print("\nMissing values:")
print(kendaraan_bermotor.isna().sum())

# Menghitung jumlah kendaraan berdasarkan jenisnya
kolom_kendaraan = [
    "mobil_penumpang",
    "bus",
    "truk",
    "sepeda_motor"
]

kendaraan_bermotor["total_hitung"] = (
    kendaraan_bermotor[kolom_kendaraan].sum(
        axis=1, min_count=4
    )
)

# Menghitung selisih dengan total dari BPS
kendaraan_bermotor["selisih"] = (
    kendaraan_bermotor["total"]
    - kendaraan_bermotor["total_hitung"]
)

print("\nData dengan total tidak sesuai:")
display(
    kendaraan_bermotor.loc[
        kendaraan_bermotor["selisih"].notna()
        & (kendaraan_bermotor["selisih"].abs() > 0.01)
    ]
)

# Validasi jumlah data
assert len(kendaraan_bermotor) == 70
assert (kendaraan_bermotor.groupby("tahun").size() == 35).all()

# Validasi kesesuaian wilayah
for tahun, df in kendaraan_bermotor.groupby("tahun"):
    assert set(df["kabupaten_kota"]) == daftar_wilayah, (
        f"Wilayah tahun {tahun} tidak sesuai"
    )

# Validasi missing values
kolom_utama = ["kabupaten_kota"] + kolom_nilai + ["tahun"]

assert kendaraan_bermotor[kolom_utama].notna().all().all()

# Validasi duplikasi
assert not kendaraan_bermotor.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

# Validasi nilai negatif
assert (kendaraan_bermotor[kolom_nilai] >= 0).all().all()

print("\nValidasi struktur berhasil!")

In [ ]:
# ======================
# Memeriksa Anomali Data Kendaraan
# ======================

# Membaca ulang data asli 2020
kendaraan_2020_asli = pd.read_csv(file_kendaraan_2020)

# Memeriksa data asli Kota Tegal
print("Data asli Kota Tegal 2020:")

display(
    kendaraan_2020_asli.loc[
        kendaraan_2020_asli.iloc[:, 0]
        .astype(str)
        .str.strip()
        .eq("Kota Tegal")
    ]
)

# Mengambil data yang totalnya tidak sesuai
kendaraan_anomali = kendaraan_bermotor.loc[
    kendaraan_bermotor["selisih"].abs() > 0.01
].copy()

print("\nData dengan total tidak sesuai:")
display(kendaraan_anomali)

print("\nJumlah anomali:", len(kendaraan_anomali))

In [ ]:
# ======================
# Validasi Wilayah Antar-Dataset
# ======================

# Mengambil nama wilayah dari setiap dataset
wilayah_laka = set(
    laka_lantas["kabupaten_kota"].astype(str).str.strip()
)

wilayah_jalan = set(
    jalan_clean["kabupaten_kota"].astype(str).str.strip()
)

wilayah_kendaraan = set(
    kendaraan_bermotor["kabupaten_kota"].astype(str).str.strip()
)

# Memeriksa jumlah wilayah
print("Jumlah wilayah kecelakaan:", len(wilayah_laka))
print("Jumlah wilayah panjang jalan:", len(wilayah_jalan))
print("Jumlah wilayah kendaraan:", len(wilayah_kendaraan))

# Memeriksa wilayah yang tidak cocok
print("\nWilayah kecelakaan yang tidak ada di panjang jalan:")
print(sorted(wilayah_laka - wilayah_jalan))

print("\nWilayah panjang jalan yang tidak ada di kecelakaan:")
print(sorted(wilayah_jalan - wilayah_laka))

print("\nWilayah kendaraan yang tidak ada di panjang jalan:")
print(sorted(wilayah_kendaraan - wilayah_jalan))

print("\nWilayah panjang jalan yang tidak ada di kendaraan:")
print(sorted(wilayah_jalan - wilayah_kendaraan))

# Validasi kesesuaian wilayah
assert len(wilayah_laka) == 35
assert len(wilayah_jalan) == 35
assert len(wilayah_kendaraan) == 35

assert wilayah_laka == wilayah_jalan
assert wilayah_jalan == wilayah_kendaraan

print("\nValidasi wilayah berhasil!")

# Membuat Dimensi

In [ ]:
# ======================
# Membuat Dimensi Tahun
# ======================

dim_tahun = pd.DataFrame({
    "id_tahun": [1, 2],
    "tahun": [2020, 2021]
})

display(dim_tahun)

In [ ]:
# ======================
# Membuat Dimensi Bulan
# ======================

daftar_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "Nopember", "Desember"
]

dim_bulan = pd.DataFrame({
    "id_bulan": range(1, 13),
    "nama_bulan": daftar_bulan
})

# Menentukan kuartal
dim_bulan["kuartal"] = (
    (dim_bulan["id_bulan"] - 1) // 3 + 1
)

display(dim_bulan)

In [ ]:
# ======================
# Membuat Dimensi Lokasi
# ======================

# Mengambil nama wilayah dari data kecelakaan
dim_lokasi = (
    laka_lantas[["kabupaten_kota"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Membersihkan format nama wilayah
dim_lokasi["kabupaten_kota"] = (
    dim_lokasi["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Membuat ID lokasi berdasarkan urutan data
dim_lokasi.insert(
    0, "id_lokasi", range(1, len(dim_lokasi) + 1)
)

display(dim_lokasi)

In [ ]:
# ======================
# Validasi Dimensi Lokasi
# ======================

print("Jumlah wilayah:", len(dim_lokasi))
print("Jumlah wilayah unik:", dim_lokasi["kabupaten_kota"].nunique())

# Memeriksa duplikasi dan missing values
assert len(dim_lokasi) == 35
assert dim_lokasi["kabupaten_kota"].is_unique
assert dim_lokasi.notna().all().all()

# Memeriksa kesesuaian wilayah antar sumber
def normalisasi_lokasi(df):
    return set(
        df["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

wilayah_dim = set(dim_lokasi["kabupaten_kota"])
wilayah_jalan = normalisasi_lokasi(jalan_clean)
wilayah_kendaraan = normalisasi_lokasi(kendaraan_bermotor)

assert wilayah_dim == wilayah_jalan
assert wilayah_dim == wilayah_kendaraan

# Memeriksa kabupaten dan kota dengan nama sama
for nama in ["magelang", "semarang", "pekalongan", "tegal"]:
    assert nama in wilayah_dim
    assert f"kota_{nama}" in wilayah_dim

print("\nValidasi dimensi lokasi berhasil!")

In [ ]:
# ======================
# Membuat Dimensi Jenis Kendaraan
# ======================

# Mengambil jenis kendaraan dari data yang sudah dibersihkan
jenis_kendaraan = [
    col for col in kendaraan_bermotor.columns
    if col in ["mobil_penumpang", "bus", "truk", "sepeda_motor"]
]

dim_jenis_kendaraan = pd.DataFrame({
    "nama_jenis_kendaraan": jenis_kendaraan
})

# Membuat ID jenis kendaraan
dim_jenis_kendaraan.insert(
    0, "id_jenis_kendaraan",
    range(1, len(dim_jenis_kendaraan) + 1)
)

display(dim_jenis_kendaraan)

In [ ]:
# ======================
# Membuat Dimensi Jenis Transportasi
# ======================

# Mengambil jenis transportasi berdasarkan sumber data
jenis_transportasi = [
    "Angkutan Laut",
    "Penerbangan Domestik"
]

dim_jenis_transportasi = pd.DataFrame({
    "nama_jenis_transportasi": jenis_transportasi
})

# Membuat ID jenis transportasi
dim_jenis_transportasi.insert(
    0, "id_jenis_transportasi",
    range(1, len(dim_jenis_transportasi) + 1)
)

display(dim_jenis_transportasi)

In [ ]:
# ======================
# Membuat Dimensi Jenis Jalan
# ======================

# Mengambil jenis jalan dari nama kolom
kolom_jalan = [
    "jalan_negara",
    "jalan_provinsi",
    "jalan_kabupaten_kota"
]

jenis_jalan = [
    col.replace("jalan_", "")
    for col in kolom_jalan
    if col in jalan_clean.columns
]

dim_jenis_jalan = pd.DataFrame({
    "nama_jenis_jalan": jenis_jalan
})

# Membuat ID jenis jalan
dim_jenis_jalan.insert(
    0, "id_jenis_jalan",
    range(1, len(dim_jenis_jalan) + 1)
)

display(dim_jenis_jalan)

In [ ]:
# ======================
# Membuat Dimensi Indikator Kecelakaan
# ======================

# Mengambil indikator dari nama kolom data kecelakaan
kolom_indikator = [
    col for col in laka_lantas.columns
    if col.endswith(("_2020", "_2021"))
]

# Menghapus akhiran tahun dan mengambil indikator unik
indikator_kecelakaan = sorted(set(
    col.rsplit("_", 1)[0]
    for col in kolom_indikator
))

dim_indikator_kecelakaan = pd.DataFrame({
    "nama_indikator": indikator_kecelakaan
})

# Membuat ID indikator
dim_indikator_kecelakaan.insert(
    0, "id_indikator",
    range(1, len(dim_indikator_kecelakaan) + 1)
)

display(dim_indikator_kecelakaan)

In [ ]:
# ======================
# Validasi Seluruh Dimensi
# ======================

daftar_dimensi = {
    "dim_tahun": (dim_tahun, 2),
    "dim_bulan": (dim_bulan, 12),
    "dim_lokasi": (dim_lokasi, 35),
    "dim_jenis_kendaraan": (dim_jenis_kendaraan, 4),
    "dim_jenis_transportasi": (dim_jenis_transportasi, 2),
    "dim_jenis_jalan": (dim_jenis_jalan, 3),
    "dim_indikator_kecelakaan": (dim_indikator_kecelakaan, 5)
}

for nama, (df, jumlah) in daftar_dimensi.items():
    # Memeriksa jumlah baris
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"

    # Memeriksa missing values
    assert df.notna().all().all(), f"Ada missing values pada {nama}"

    # Memeriksa ID dimensi
    assert df.iloc[:, 0].is_unique, f"ID duplikat pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

print("\nValidasi seluruh dimensi berhasil!")

# Membuat Fakta

In [ ]:
# ======================
# Membuat Fakta Transportasi
# ======================

# Menyiapkan data penerbangan domestik
pesawat_fact = pesawat_clean[
    ["tahun", "bulan", "penumpang"]
].copy()

pesawat_fact["jenis_transportasi"] = "Penerbangan Domestik"

# Menyiapkan data angkutan laut
laut_fact = debarkasi_angkutan_laut[
    ["tahun", "bulan", "penumpang"]
].copy()

laut_fact["jenis_transportasi"] = "Angkutan Laut"

# Menggabungkan kedua data
transportasi = pd.concat(
    [laut_fact, pesawat_fact],
    ignore_index=True
)

# Mengubah tipe data
transportasi["tahun"] = transportasi["tahun"].astype(int)
transportasi["penumpang"] = pd.to_numeric(
    transportasi["penumpang"],
    errors="raise"
)

display(transportasi.head(10))

In [ ]:
# ======================
# Mapping Dimensi Fakta Transportasi
# ======================

# Menghubungkan dengan dimensi tahun
fakta_transportasi = transportasi.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi bulan
fakta_transportasi = fakta_transportasi.merge(
    dim_bulan[["id_bulan", "nama_bulan"]],
    left_on="bulan",
    right_on="nama_bulan",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi jenis transportasi
fakta_transportasi = fakta_transportasi.merge(
    dim_jenis_transportasi,
    left_on="jenis_transportasi",
    right_on="nama_jenis_transportasi",
    how="left",
    validate="many_to_one"
)

# Mengambil kolom sesuai struktur fact table
fakta_transportasi = fakta_transportasi[
    [
        "id_bulan",
        "id_tahun",
        "id_jenis_transportasi",
        "penumpang"
    ]
].copy()

# Mengurutkan data
fakta_transportasi = fakta_transportasi.sort_values(
    ["id_tahun", "id_jenis_transportasi", "id_bulan"]
).reset_index(drop=True)

display(fakta_transportasi)

In [ ]:
# ======================
# Validasi Fakta Transportasi
# ======================

print("Jumlah baris:", len(fakta_transportasi))

print("\nMissing values:")
print(fakta_transportasi.isna().sum())

print("\nJumlah data per tahun dan jenis transportasi:")
print(
    fakta_transportasi.groupby(
        ["id_tahun", "id_jenis_transportasi"]
    ).size()
)

# Memastikan jumlah data lengkap
assert len(fakta_transportasi) == 48

# Memastikan setiap kombinasi tahun dan moda memiliki 12 bulan
assert (
    fakta_transportasi.groupby(
        ["id_tahun", "id_jenis_transportasi"]
    )["id_bulan"].nunique() == 12
).all()

# Memastikan seluruh foreign key ditemukan
assert fakta_transportasi[
    ["id_bulan", "id_tahun", "id_jenis_transportasi"]
].notna().all().all()

# Memastikan tidak ada duplikasi
assert not fakta_transportasi.duplicated(
    subset=["id_bulan", "id_tahun", "id_jenis_transportasi"]
).any()

# Memastikan penumpang valid
assert fakta_transportasi["penumpang"].notna().all()
assert (fakta_transportasi["penumpang"] >= 0).all()

# Memastikan hasil angkutan laut sesuai BPS
total_laut = (
    fakta_transportasi[
        fakta_transportasi["id_jenis_transportasi"] == 1
    ]
    .merge(dim_tahun, on="id_tahun")
    .groupby("tahun")["penumpang"]
    .sum()
)

assert total_laut.loc[2020] == 133604
assert total_laut.loc[2021] == 152016

print("\nValidasi fakta transportasi berhasil!")

In [ ]:
# ======================
# Membuat Fakta Kecelakaan
# ======================

# Mengubah data kecelakaan menjadi format long
laka_long = laka_lantas.melt(
    id_vars=["kabupaten_kota"],
    var_name="indikator_tahun",
    value_name="nilai"
)

# Memisahkan indikator dan tahun
laka_long["tahun"] = (
    laka_long["indikator_tahun"]
    .str.extract(r"_(2020|2021)$")[0]
    .astype(int)
)

laka_long["indikator"] = (
    laka_long["indikator_tahun"]
    .str.replace(r"_(2020|2021)$", "", regex=True)
)

# Menyamakan format nama wilayah
laka_long["kabupaten_kota"] = (
    laka_long["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Mengubah nilai menjadi numerik
laka_long["nilai"] = pd.to_numeric(
    laka_long["nilai"],
    errors="raise"
)

# Memilih kolom yang diperlukan
laka_long = laka_long[
    ["kabupaten_kota", "tahun", "indikator", "nilai"]
]

display(laka_long.head(10))
print("Jumlah baris:", len(laka_long))

In [ ]:
# ======================
# Mapping Dimensi Fakta Kecelakaan
# ======================

# Menghubungkan dengan dimensi lokasi
fakta_kecelakaan = laka_long.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
fakta_kecelakaan = fakta_kecelakaan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi indikator
fakta_kecelakaan = fakta_kecelakaan.merge(
    dim_indikator_kecelakaan,
    left_on="indikator",
    right_on="nama_indikator",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh foreign key ditemukan
assert fakta_kecelakaan[
    ["id_lokasi", "id_tahun", "id_indikator"]
].notna().all().all(), "Ada data yang gagal mapping dimensi"

# Memilih kolom sesuai struktur fact table
fakta_kecelakaan = fakta_kecelakaan[
    ["id_lokasi", "id_tahun", "id_indikator", "nilai"]
].copy()

# Mengubah tipe data foreign key menjadi integer
for col in ["id_lokasi", "id_tahun", "id_indikator"]:
    fakta_kecelakaan[col] = fakta_kecelakaan[col].astype(int)

# Mengurutkan data
fakta_kecelakaan = fakta_kecelakaan.sort_values(
    ["id_tahun", "id_indikator", "id_lokasi"]
).reset_index(drop=True)

display(fakta_kecelakaan)

In [ ]:
# ======================
# Validasi Fakta Kecelakaan
# ======================

print("Jumlah baris:", len(fakta_kecelakaan))

print("\nMissing values:")
print(fakta_kecelakaan.isna().sum())

print("\nJumlah data per tahun:")
print(
    fakta_kecelakaan.groupby("id_tahun").size()
)

print("\nJumlah data per indikator:")
print(
    fakta_kecelakaan.groupby("id_indikator").size()
)

# Validasi jumlah data
assert len(fakta_kecelakaan) == 350

# Validasi missing values
assert fakta_kecelakaan.notna().all().all()

# Validasi duplikasi kombinasi foreign key
assert not fakta_kecelakaan.duplicated(
    subset=["id_lokasi", "id_tahun", "id_indikator"]
).any()

# Validasi jumlah wilayah, tahun, dan indikator
assert fakta_kecelakaan["id_lokasi"].nunique() == 35
assert fakta_kecelakaan["id_tahun"].nunique() == 2
assert fakta_kecelakaan["id_indikator"].nunique() == 5

# Validasi setiap kombinasi tahun dan indikator
assert (
    fakta_kecelakaan.groupby(
        ["id_tahun", "id_indikator"]
    )["id_lokasi"].nunique() == 35
).all()

# Validasi nilai kecelakaan
assert (fakta_kecelakaan["nilai"] >= 0).all()

# Memastikan total nilai tidak berubah setelah mapping
assert np.isclose(
    laka_long["nilai"].sum(),
    fakta_kecelakaan["nilai"].sum()
)

# Memastikan kabupaten dan kota memiliki ID berbeda
for nama in ["magelang", "semarang", "pekalongan", "tegal"]:
    id_kabupaten = dim_lokasi.loc[
        dim_lokasi["kabupaten_kota"] == nama,
        "id_lokasi"
    ].iloc[0]

    id_kota = dim_lokasi.loc[
        dim_lokasi["kabupaten_kota"] == f"kota_{nama}",
        "id_lokasi"
    ].iloc[0]

    assert id_kabupaten != id_kota

    print(f"{nama}: Kabupaten ID {id_kabupaten}, Kota ID {id_kota}")

print("\nValidasi fakta kecelakaan berhasil!")

In [ ]:
# ======================
# Membuat Fakta Kepadatan
# ======================

# Mengambil jenis kendaraan dari dimensi
kolom_kendaraan = dim_jenis_kendaraan[
    "nama_jenis_kendaraan"
].tolist()

# Mengubah data kendaraan dari wide ke long
kendaraan_long = kendaraan_bermotor.melt(
    id_vars=["kabupaten_kota", "tahun"],
    value_vars=kolom_kendaraan,
    var_name="jenis_kendaraan",
    value_name="total_kendaraan"
)

# Mengambil data panjang jalan
jalan_fact = jalan_clean[
    ["kabupaten_kota", "tahun", "total_panjang_jalan"]
].copy()

# Menyamakan format nama wilayah
for df in [kendaraan_long, jalan_fact]:
    df["kabupaten_kota"] = (
        df["kabupaten_kota"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    df["tahun"] = df["tahun"].astype(int)

# Memastikan data panjang jalan tidak duplikat
assert not jalan_fact.duplicated(
    subset=["kabupaten_kota", "tahun"]
).any()

# Menggabungkan kendaraan dan panjang jalan
kepadatan = kendaraan_long.merge(
    jalan_fact,
    on=["kabupaten_kota", "tahun"],
    how="left",
    validate="many_to_one"
)

display(kepadatan.head(10))
print("Jumlah baris:", len(kepadatan))

In [ ]:
# ======================
# Mapping Dimensi Fakta Kepadatan
# ======================

# Menghubungkan dengan dimensi lokasi
fakta_kepadatan = kepadatan.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
fakta_kepadatan = fakta_kepadatan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi jenis kendaraan
fakta_kepadatan = fakta_kepadatan.merge(
    dim_jenis_kendaraan,
    left_on="jenis_kendaraan",
    right_on="nama_jenis_kendaraan",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh mapping berhasil
assert fakta_kepadatan[
    [
        "id_lokasi",
        "id_tahun",
        "id_jenis_kendaraan",
        "total_panjang_jalan"
    ]
].notna().all().all()

# Memastikan panjang jalan valid
assert (fakta_kepadatan["total_panjang_jalan"] > 0).all()

# Menghitung kendaraan per kilometer
fakta_kepadatan["kendaraan_per_km"] = (
    fakta_kepadatan["total_kendaraan"]
    / fakta_kepadatan["total_panjang_jalan"]
)

# Memilih kolom fact table
fakta_kepadatan = fakta_kepadatan[
    [
        "id_lokasi",
        "id_tahun",
        "id_jenis_kendaraan",
        "total_kendaraan",
        "total_panjang_jalan",
        "kendaraan_per_km"
    ]
].copy()

# Mengubah foreign key menjadi integer
for col in ["id_lokasi", "id_tahun", "id_jenis_kendaraan"]:
    fakta_kepadatan[col] = fakta_kepadatan[col].astype(int)

# Mengurutkan data
fakta_kepadatan = fakta_kepadatan.sort_values(
    ["id_tahun", "id_lokasi", "id_jenis_kendaraan"]
).reset_index(drop=True)

display(fakta_kepadatan)

In [ ]:
# ======================
# Validasi Fakta Kepadatan
# ======================

print("Jumlah baris:", len(fakta_kepadatan))

print("\nMissing values:")
print(fakta_kepadatan.isna().sum())

print("\nJumlah data per tahun:")
print(fakta_kepadatan.groupby("id_tahun").size())

print("\nJumlah data per jenis kendaraan:")
print(fakta_kepadatan.groupby("id_jenis_kendaraan").size())

# Validasi jumlah baris
assert len(fakta_kepadatan) == 280

# Validasi missing values
assert fakta_kepadatan.notna().all().all()

# Validasi duplikasi
assert not fakta_kepadatan.duplicated(
    subset=["id_lokasi", "id_tahun", "id_jenis_kendaraan"]
).any()

# Validasi setiap wilayah memiliki 4 jenis kendaraan per tahun
assert (
    fakta_kepadatan.groupby(
        ["id_lokasi", "id_tahun"]
    )["id_jenis_kendaraan"].nunique() == 4
).all()

# Validasi nilai
assert (fakta_kepadatan["total_kendaraan"] >= 0).all()
assert (fakta_kepadatan["total_panjang_jalan"] > 0).all()
assert np.isfinite(fakta_kepadatan["kendaraan_per_km"]).all()

# Memastikan jumlah kendaraan tidak berubah
assert np.isclose(
    kendaraan_long["total_kendaraan"].sum(),
    fakta_kepadatan["total_kendaraan"].sum()
)

# Memastikan perhitungan rasio benar
assert np.allclose(
    fakta_kepadatan["kendaraan_per_km"],
    fakta_kepadatan["total_kendaraan"]
    / fakta_kepadatan["total_panjang_jalan"]
)

print("\nValidasi fakta kepadatan berhasil!")

In [ ]:
# ======================
# Membuat Fakta Risiko
# ======================

# Mengambil ID indikator jumlah kecelakaan
id_kecelakaan = dim_indikator_kecelakaan.loc[
    dim_indikator_kecelakaan["nama_indikator"] == "jumlah_kecelakaan",
    "id_indikator"
].iloc[0]

# Mengambil jumlah kecelakaan dari fakta_kecelakaan
risiko_laka = fakta_kecelakaan.loc[
    fakta_kecelakaan["id_indikator"] == id_kecelakaan,
    ["id_lokasi", "id_tahun", "nilai"]
].copy()

# Mengubah nama kolom
risiko_laka = risiko_laka.rename(
    columns={"nilai": "jumlah_kecelakaan"}
)

# Mengambil total panjang jalan
risiko_jalan = jalan_clean[
    ["kabupaten_kota", "tahun", "total_panjang_jalan"]
].copy()

# Menyamakan format nama wilayah
risiko_jalan["kabupaten_kota"] = (
    risiko_jalan["kabupaten_kota"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Menghubungkan panjang jalan dengan dimensi lokasi
risiko_jalan = risiko_jalan.merge(
    dim_lokasi,
    on="kabupaten_kota",
    how="left",
    validate="many_to_one"
)

# Menghubungkan dengan dimensi tahun
risiko_jalan = risiko_jalan.merge(
    dim_tahun,
    on="tahun",
    how="left",
    validate="many_to_one"
)

# Memastikan seluruh wilayah dan tahun berhasil dipetakan
assert risiko_jalan[
    ["id_lokasi", "id_tahun"]
].notna().all().all()

# Menggabungkan data kecelakaan dan panjang jalan
risiko = risiko_laka.merge(
    risiko_jalan[
        ["id_lokasi", "id_tahun", "total_panjang_jalan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="left",
    validate="one_to_one"
)

display(risiko.head(10))
print("Jumlah baris:", len(risiko))

In [ ]:
# ======================
# Menghitung Fakta Risiko
# ======================

# Memastikan data panjang jalan tersedia dan valid
assert risiko["total_panjang_jalan"].notna().all()
assert (risiko["total_panjang_jalan"] > 0).all()

# Menghitung rasio kecelakaan per kilometer
risiko["rasio_kecelakaan_per_km"] = (
    risiko["jumlah_kecelakaan"]
    / risiko["total_panjang_jalan"]
)

# Membuat tabel fakta risiko
fakta_risiko = risiko[
    [
        "id_lokasi",
        "id_tahun",
        "jumlah_kecelakaan",
        "total_panjang_jalan",
        "rasio_kecelakaan_per_km"
    ]
].copy()

# Mengubah foreign key menjadi integer
for col in ["id_lokasi", "id_tahun"]:
    fakta_risiko[col] = fakta_risiko[col].astype(int)

# Mengurutkan data
fakta_risiko = fakta_risiko.sort_values(
    ["id_tahun", "id_lokasi"]
).reset_index(drop=True)

display(fakta_risiko)

In [ ]:
# ======================
# Validasi Fakta Risiko
# ======================

print("Jumlah baris:", len(fakta_risiko))

print("\nMissing values:")
print(fakta_risiko.isna().sum())

print("\nJumlah data per tahun:")
print(fakta_risiko.groupby("id_tahun").size())

print("\nJumlah wilayah:")
print(fakta_risiko["id_lokasi"].nunique())

# Validasi jumlah baris
assert len(fakta_risiko) == 70

# Validasi missing values
assert fakta_risiko.notna().all().all()

# Validasi duplikasi
assert not fakta_risiko.duplicated(
    subset=["id_lokasi", "id_tahun"]
).any()

# Memastikan tiap tahun memiliki 35 wilayah
assert (
    fakta_risiko.groupby("id_tahun")["id_lokasi"].nunique()
    == 35
).all()

# Validasi nilai
assert (fakta_risiko["jumlah_kecelakaan"] >= 0).all()
assert (fakta_risiko["total_panjang_jalan"] > 0).all()

# Validasi hasil perhitungan
assert np.isfinite(
    fakta_risiko["rasio_kecelakaan_per_km"]
).all()

assert np.allclose(
    fakta_risiko["rasio_kecelakaan_per_km"],
    fakta_risiko["jumlah_kecelakaan"]
    / fakta_risiko["total_panjang_jalan"]
)

# Memastikan jumlah kecelakaan tidak berubah
assert np.isclose(
    risiko_laka["jumlah_kecelakaan"].sum(),
    fakta_risiko["jumlah_kecelakaan"].sum()
)

print("\nValidasi fakta risiko berhasil!")

# Validasi Akhir Hasil ETL

In [ ]:
# ======================
# Validasi Struktur Seluruh Tabel
# ======================

# Daftar tabel dimensi dan jumlah baris yang diharapkan
tabel_dimensi = {
    "dim_tahun": (dim_tahun, 2, "id_tahun"),
    "dim_bulan": (dim_bulan, 12, "id_bulan"),
    "dim_lokasi": (dim_lokasi, 35, "id_lokasi"),
    "dim_jenis_kendaraan": (dim_jenis_kendaraan, 4, "id_jenis_kendaraan"),
    "dim_jenis_transportasi": (dim_jenis_transportasi, 2, "id_jenis_transportasi"),
    "dim_jenis_jalan": (dim_jenis_jalan, 3, "id_jenis_jalan"),
    "dim_indikator_kecelakaan": (dim_indikator_kecelakaan, 5, "id_indikator")
}

# Daftar tabel fakta dan jumlah baris yang diharapkan
tabel_fakta = {
    "fakta_transportasi": (fakta_transportasi, 48),
    "fakta_kecelakaan": (fakta_kecelakaan, 350),
    "fakta_kepadatan": (fakta_kepadatan, 280),
    "fakta_risiko": (fakta_risiko, 70)
}

# Validasi tabel dimensi
for nama, (df, jumlah, id_col) in tabel_dimensi.items():
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"
    assert df.notna().all().all(), f"Missing values pada {nama}"
    assert df[id_col].is_unique, f"ID duplikat pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

print()

# Validasi tabel fakta
for nama, (df, jumlah) in tabel_fakta.items():
    assert len(df) == jumlah, f"Jumlah baris {nama} tidak sesuai"
    assert df.notna().all().all(), f"Missing values pada {nama}"

    print(f"{nama}: {len(df)} baris - OK")

# Menghitung total baris
total_dimensi = sum(len(df) for df, _, _ in tabel_dimensi.values())
total_fakta = sum(len(df) for df, _ in tabel_fakta.values())

print("\nTotal baris dimensi:", total_dimensi)
print("Total baris fakta:", total_fakta)
print("Total seluruh baris:", total_dimensi + total_fakta)

assert total_dimensi + total_fakta == 811

print("\nValidasi struktur seluruh tabel berhasil!")

In [ ]:
# ======================
# Validasi Foreign Key
# ======================

# Daftar hubungan antara tabel fakta dan dimensi
relasi_fk = [
    (fakta_transportasi, "id_bulan", dim_bulan),
    (fakta_transportasi, "id_tahun", dim_tahun),
    (fakta_transportasi, "id_jenis_transportasi", dim_jenis_transportasi),

    (fakta_kecelakaan, "id_lokasi", dim_lokasi),
    (fakta_kecelakaan, "id_tahun", dim_tahun),
    (fakta_kecelakaan, "id_indikator", dim_indikator_kecelakaan),

    (fakta_kepadatan, "id_lokasi", dim_lokasi),
    (fakta_kepadatan, "id_tahun", dim_tahun),
    (fakta_kepadatan, "id_jenis_kendaraan", dim_jenis_kendaraan),

    (fakta_risiko, "id_lokasi", dim_lokasi),
    (fakta_risiko, "id_tahun", dim_tahun)
]

# Memeriksa seluruh foreign key
for df_fakta, kolom_fk, df_dimensi in relasi_fk:

    id_valid = set(df_dimensi[kolom_fk])

    id_fakta = set(df_fakta[kolom_fk])

    id_tidak_valid = id_fakta - id_valid

    assert len(id_tidak_valid) == 0, (
        f"Foreign key {kolom_fk} tidak valid: {id_tidak_valid}"
    )

    print(f"{kolom_fk} - OK")

print("\nSeluruh foreign key valid!")

In [ ]:
# ======================
# Validasi Konsistensi Antar-Fakta
# ======================

# 1. Memeriksa panjang jalan pada fakta kepadatan

cek_panjang = fakta_kepadatan.groupby(
    ["id_lokasi", "id_tahun"]
).agg(
    panjang_jalan=("total_panjang_jalan", "first"),
    jumlah_nilai_panjang=("total_panjang_jalan", "nunique")
).reset_index()

# Setiap wilayah-tahun harus memakai satu nilai panjang jalan
assert (cek_panjang["jumlah_nilai_panjang"] == 1).all()

# Membandingkan dengan fakta risiko
cek_panjang = cek_panjang.merge(
    fakta_risiko[
        ["id_lokasi", "id_tahun", "total_panjang_jalan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert len(cek_panjang) == 70
assert (cek_panjang["_merge"] == "both").all()

assert np.allclose(
    cek_panjang["panjang_jalan"],
    cek_panjang["total_panjang_jalan"]
)

print("Konsistensi panjang jalan - OK")


# 2. Memeriksa jumlah kecelakaan pada fakta risiko

id_kecelakaan = dim_indikator_kecelakaan.loc[
    dim_indikator_kecelakaan["nama_indikator"] == "jumlah_kecelakaan",
    "id_indikator"
].iloc[0]

cek_kecelakaan = fakta_kecelakaan.loc[
    fakta_kecelakaan["id_indikator"] == id_kecelakaan,
    ["id_lokasi", "id_tahun", "nilai"]
].copy()

cek_kecelakaan = cek_kecelakaan.rename(
    columns={"nilai": "jumlah_kecelakaan_asli"}
)

cek_kecelakaan = cek_kecelakaan.merge(
    fakta_risiko[
        ["id_lokasi", "id_tahun", "jumlah_kecelakaan"]
    ],
    on=["id_lokasi", "id_tahun"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert len(cek_kecelakaan) == 70
assert (cek_kecelakaan["_merge"] == "both").all()

assert np.allclose(
    cek_kecelakaan["jumlah_kecelakaan_asli"],
    cek_kecelakaan["jumlah_kecelakaan"]
)

print("Konsistensi jumlah kecelakaan - OK")


# 3. Memeriksa kelengkapan fakta transportasi

cek_transportasi = fakta_transportasi.groupby(
    ["id_tahun", "id_jenis_transportasi"]
).agg(
    jumlah_baris=("id_bulan", "size"),
    jumlah_bulan=("id_bulan", "nunique")
).reset_index()

assert len(cek_transportasi) == 4
assert (cek_transportasi["jumlah_baris"] == 12).all()
assert (cek_transportasi["jumlah_bulan"] == 12).all()

print("Kelengkapan transportasi - OK")

print("\nValidasi konsistensi antar-fakta berhasil!")

# Connect ke MySQL

In [ ]:
# ======================
# Koneksi ke Database MySQL
# ======================

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from getpass import getpass

# Konfigurasi database
host = "127.0.0.1"
port = 3306
user = "root"
password = getpass("Password MySQL (Enter jika kosong): ")
database = "tugas_iak"

# Membuat URL koneksi
db_url = URL.create(
    drivername="mysql+pymysql",
    username=user,
    password=password,
    host=host,
    port=port,
    database=database
)

# Membuat engine
engine = create_engine(
    db_url,
    pool_pre_ping=True
)

# Memeriksa koneksi
with engine.connect() as conn:
    nama_database = conn.execute(
        text("SELECT DATABASE()")
    ).scalar()

    versi_mysql = conn.execute(
        text("SELECT VERSION()")
    ).scalar()

print("Koneksi MySQL berhasil!")
print("Database:", nama_database)
print("Versi MySQL:", versi_mysql)

assert nama_database == "tugas_iak"

# Loading ke Mysql

In [ ]:
# ======================
# Menyiapkan Data untuk Loading
# ======================

# Daftar tabel dan dataframe yang akan dimasukkan
data_load = {
    "dim_tahun": dim_tahun,
    "dim_bulan": dim_bulan,
    "dim_lokasi": dim_lokasi,
    "dim_jenis_kendaraan": dim_jenis_kendaraan,
    "dim_jenis_transportasi": dim_jenis_transportasi,
    "dim_jenis_jalan": dim_jenis_jalan,
    "dim_indikator_kecelakaan": dim_indikator_kecelakaan,

    "fakta_transportasi": fakta_transportasi,
    "fakta_kecelakaan": fakta_kecelakaan,
    "fakta_kepadatan": fakta_kepadatan,
    "fakta_risiko": fakta_risiko
}

# Kolom yang akan dimasukkan ke masing-masing tabel
kolom_load = {
    "dim_tahun": ["id_tahun", "tahun"],
    "dim_bulan": ["id_bulan", "nama_bulan", "kuartal"],
    "dim_lokasi": ["id_lokasi", "kabupaten_kota"],
    "dim_jenis_kendaraan": [
        "id_jenis_kendaraan", "nama_jenis_kendaraan"
    ],
    "dim_jenis_transportasi": [
        "id_jenis_transportasi", "nama_jenis_transportasi"
    ],
    "dim_jenis_jalan": [
        "id_jenis_jalan", "nama_jenis_jalan"
    ],
    "dim_indikator_kecelakaan": [
        "id_indikator", "nama_indikator"
    ],

    "fakta_transportasi": [
        "id_bulan", "id_tahun",
        "id_jenis_transportasi", "penumpang"
    ],
    "fakta_kecelakaan": [
        "id_lokasi", "id_tahun",
        "id_indikator", "nilai"
    ],
    "fakta_kepadatan": [
        "id_lokasi", "id_tahun",
        "id_jenis_kendaraan", "total_kendaraan",
        "total_panjang_jalan", "kendaraan_per_km"
    ],
    "fakta_risiko": [
        "id_lokasi", "id_tahun",
        "jumlah_kecelakaan",
        "total_panjang_jalan",
        "rasio_kecelakaan_per_km"
    ]
}

# Validasi kesiapan data
for nama, df in data_load.items():

    assert set(df.columns) == set(kolom_load[nama]), (
        f"Kolom {nama} tidak sesuai"
    )

    assert df[kolom_load[nama]].notna().all().all(), (
        f"Ada missing values pada {nama}"
    )

    print(f"{nama}: {len(df)} baris siap dimuat")

print("\nSeluruh data siap untuk loading!")

In [ ]:
# ======================
# Loading Data ke MySQL
# ======================

# Memastikan seluruh tabel sudah ada
tabel_mysql = set(inspect(engine).get_table_names())

assert set(data_load).issubset(tabel_mysql), (
    "Ada tabel data warehouse yang belum tersedia di MySQL"
)

# Memulai transaksi loading
with engine.begin() as conn:

    # Memastikan database yang digunakan benar
    assert conn.execute(
        text("SELECT DATABASE()")
    ).scalar() == "tugas_iak"

    # Memastikan seluruh tabel masih kosong
    for nama in data_load:

        jumlah_awal = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        assert jumlah_awal == 0, (
            f"Tabel {nama} belum kosong. "
            "Jangan loading dua kali tanpa reset."
        )

    # Memasukkan data dimensi terlebih dahulu,
    # kemudian data fakta
    for nama, df in data_load.items():

        df[kolom_load[nama]].to_sql(
            name=nama,
            con=conn,
            if_exists="append",
            index=False,
            chunksize=500,
            method="multi"
        )

        # Memeriksa jumlah baris sebelum commit
        jumlah_mysql = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        assert jumlah_mysql == len(df), (
            f"Jumlah baris {nama} tidak sesuai"
        )

        print(f"{nama}: {jumlah_mysql} baris berhasil dimuat")

print("\nLoading seluruh data warehouse berhasil!")

In [ ]:
# ======================
# Verifikasi Jumlah Baris MySQL
# ======================

hasil_verifikasi = []

with engine.connect() as conn:

    for nama, df in data_load.items():

        jumlah_mysql = conn.execute(
            text(f"SELECT COUNT(*) FROM `{nama}`")
        ).scalar()

        hasil_verifikasi.append({
            "nama_tabel": nama,
            "baris_python": len(df),
            "baris_mysql": jumlah_mysql,
            "status": (
                "Sesuai"
                if len(df) == jumlah_mysql
                else "Tidak Sesuai"
            )
        })

df_verifikasi = pd.DataFrame(hasil_verifikasi)

display(df_verifikasi)

assert (
    df_verifikasi["baris_python"]
    == df_verifikasi["baris_mysql"]
).all()

print("\nTotal baris MySQL:",
      df_verifikasi["baris_mysql"].sum())

assert df_verifikasi["baris_mysql"].sum() == 811

print("\nVerifikasi jumlah baris berhasil!")

In [ ]:
# ======================
# Verifikasi Isi Data MySQL
# ======================

# Kolom kunci untuk mengurutkan data
kunci_tabel = {
    "dim_tahun": ["id_tahun"],
    "dim_bulan": ["id_bulan"],
    "dim_lokasi": ["id_lokasi"],
    "dim_jenis_kendaraan": ["id_jenis_kendaraan"],
    "dim_jenis_transportasi": ["id_jenis_transportasi"],
    "dim_jenis_jalan": ["id_jenis_jalan"],
    "dim_indikator_kecelakaan": ["id_indikator"],

    "fakta_transportasi": [
        "id_tahun", "id_jenis_transportasi", "id_bulan"
    ],
    "fakta_kecelakaan": [
        "id_tahun", "id_indikator", "id_lokasi"
    ],
    "fakta_kepadatan": [
        "id_tahun", "id_lokasi", "id_jenis_kendaraan"
    ],
    "fakta_risiko": [
        "id_tahun", "id_lokasi"
    ]
}

# Kolom FLOAT dan jumlah angka desimal untuk pembacaan SQL
# 0 untuk jumlah bilangan bulat, 8 untuk nilai desimal
kolom_float = {
    "fakta_transportasi": {
        "penumpang": 0
    },
    "fakta_kecelakaan": {
        "nilai": 8
    },
    "fakta_kepadatan": {
        "total_kendaraan": 0,
        "total_panjang_jalan": 8,
        "kendaraan_per_km": 8
    },
    "fakta_risiko": {
        "jumlah_kecelakaan": 0,
        "total_panjang_jalan": 8,
        "rasio_kecelakaan_per_km": 8
    }
}

# Membandingkan isi data Python dan MySQL
with engine.connect() as conn:

    for nama, df in data_load.items():

        # Menyiapkan query SQL
        kolom_sql = []

        for col in kolom_load[nama]:

            if col in kolom_float.get(nama, {}):
                desimal = kolom_float[nama][col]

                kolom_sql.append(
                    f"CAST(`{col}` AS DECIMAL(25,{desimal})) AS `{col}`"
                )
            else:
                kolom_sql.append(f"`{col}`")

        kolom_sql = ", ".join(kolom_sql)

        # Membaca data dari MySQL
        df_mysql = pd.read_sql_query(
            text(f"SELECT {kolom_sql} FROM `{nama}`"),
            conn
        )

        # Mengubah hasil CAST menjadi numerik
        for col in kolom_float.get(nama, {}):
            df_mysql[col] = pd.to_numeric(df_mysql[col])

        # Mengurutkan data Python
        df_python = (
            df[kolom_load[nama]]
            .sort_values(kunci_tabel[nama])
            .reset_index(drop=True)
        )

        # Mengurutkan data MySQL
        df_mysql = (
            df_mysql[kolom_load[nama]]
            .sort_values(kunci_tabel[nama])
            .reset_index(drop=True)
        )

        # Memastikan jumlah baris sesuai
        assert len(df_python) == len(df_mysql), (
            f"Jumlah baris {nama} tidak sesuai"
        )

        # Memastikan nilai bilangan bulat sama persis
        for col, desimal in kolom_float.get(nama, {}).items():
            if desimal == 0:
                assert np.array_equal(
                    df_python[col].to_numpy(),
                    df_mysql[col].to_numpy()
                ), f"Nilai {col} pada {nama} tidak sesuai"

        # Membandingkan isi data
        pd.testing.assert_frame_equal(
            df_python,
            df_mysql,
            check_dtype=False,
            check_exact=False,
            rtol=1e-6,
            atol=1e-4
        )

        print(f"{nama}: {len(df_mysql)} baris - OK")

print("\nVerifikasi seluruh isi data MySQL berhasil!")

In [ ]:
# ======================
# Memeriksa Satuan Kerugian Materiil
# ======================

# Membaca data kecelakaan asli
laka_asli = pd.read_excel(
    file_laka,
    sheet_name=0,
    header=None
)

# Memeriksa header dan angka sumber
print("Header sumber:")
display(laka_asli.iloc[:8, 19:23])

print("\nNilai kerugian Cilacap dari sumber:")
print(laka_asli.iloc[7, 21])

print("\nNilai kerugian Cilacap setelah cleaning:")
display(
    laka_lantas.loc[
        laka_lantas["kabupaten_kota"]
        .astype(str).str.strip().str.lower() == "cilacap",
        ["kerugian_2020", "kerugian_2021"]
    ]
)